In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
pd.set_option("display.max_columns", None)

In [ ]:
# Volle CSV laden — nur die fuer diese Spaltengruppe noetigen Spalten (inkl. Quell-Varianten fuer die Merges).
# Bewusst die VOLLE Datei (nicht der >6%-Datensatz): cities_tags / owner / traces* liegen unter 6 %
# und fehlen in openfoodfacts_ueber6prozent.csv.zip.
CSV_PATH = "../en.openfoodfacts.org.products 2.csv"
COLS = [
    "product_name", "abbreviated_product_name", "generic_name",
    "packaging_en", "packaging", "packaging_tags", "packaging_text",
    "additives_en", "additives_tags", "additives",
    "ingredients_tags", "ingredients_text",
    "manufacturing_places_tags", "manufacturing_places",
    "ingredients_analysis_tags", "nova_group", "nutrient_levels_tags",
    "cities_tags", "owner", "traces", "traces_tags", "traces_en",
]

df = pd.read_csv(CSV_PATH, sep="\t", usecols=COLS, low_memory=False, on_bad_lines="skip")
df = df.copy()

# ── Helper ────────────────────────────────────────────────────────────────────
# Junk-Platzhalter -> fehlend. Behaelt bewusst Praefixe wie "en:"/"fr:".
INVALID = {"?", ".", ",", "n-a", "na", "none", "null", "0", "en:null", "en:none"}

def clean_series(s):
    s = s.astype("string").str.strip()
    s = s.replace("", pd.NA)
    s = s.where(~s.str.lower().isin(INVALID), pd.NA)
    s = s.where(~s.str.fullmatch(r"[?,.\-/ ]+", na=False), pd.NA)
    return s

_W = 68

def section_header(title):
    print(f"\n{'═' * _W}")
    print(f"  {title}")
    print(f"{'─' * _W}")

def merge_report(col_name, before, after):
    pct   = after / len(df) * 100
    added = after - before
    bar   = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"  {col_name:<26}  [{bar}]  {pct:5.1f}%   befuellt {after:,}  (+{added:,})")

def clean_report(col_name, before, after):
    pct     = after / len(df) * 100
    removed = before - after
    bar     = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"  {col_name:<26}  [{bar}]  {pct:5.1f}%   befuellt {after:,}  (entfernt {removed:,})")

# Taxonomie-Tags -> lesbare Form, Praefixe ENTFERNT (nur fuer packaging, wie im Hauptnotebook)
def normalize_tags(s):
    return (
        clean_series(s.astype(str).where(s.notna(), np.nan))
        .str.replace(r"\b[a-z]{2}:", "", regex=True)
        .str.replace(r"[-_]", " ", regex=True)
        .str.strip()
        .str.title()
    )


# ════════════════════════════════════════════════════════════════════════════════
# TEIL A — Reproduktion der bereits gereinigten Spalten (Logik aus Meilestein2_DatenBereinigung)
# ════════════════════════════════════════════════════════════════════════════════
section_header("Teil A  ·  product_name · packaging · additives · ingredients · manufacturing")

# Product Name — erst aus Kurz-/Oberbegriff auffuellen (Deniz), dann Text saeubern (uber6prozent)
product_name_before = df["product_name"].notna().sum()
df["product_name"] = df["product_name"].fillna(df["abbreviated_product_name"]).fillna(df["generic_name"])
df["product_name"] = (
    clean_series(df["product_name"])
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace("", pd.NA)
)
merge_report("product_name", product_name_before, df["product_name"].notna().sum())
df = df.drop(columns=["abbreviated_product_name", "generic_name"])

# Packaging — REINHEIT: nur strukturierte Quellen, Praefixe -> lesbar (entfernt). Wie Hauptnotebook.
packaging_before = df["packaging_en"].notna().sum()
df["packaging_en"] = clean_series(df["packaging_en"].astype(str).where(df["packaging_en"].notna(), np.nan))
df["packaging_en"] = df["packaging_en"].fillna(normalize_tags(df["packaging_tags"]))
merge_report("packaging_en", packaging_before, df["packaging_en"].notna().sum())
df = df.drop(columns=["packaging", "packaging_tags", "packaging_text"])

# Additives — lesbare englische Spalte als Survivor, aus Tags auffuellen, Tags verwerfen
additives_before = df["additives_en"].notna().sum()
df["additives_en"] = clean_series(df["additives_en"].astype(str).where(df["additives_en"].notna(), np.nan))
df["additives_en"] = df["additives_en"].fillna(df["additives_tags"])
merge_report("additives_en", additives_before, df["additives_en"].notna().sum())
df = df.drop(columns=["additives_tags", "additives"])

# Ingredients — Tags sauber & zaehlbar halten; ingredients_text als eigene Spalte ERHALTEN
df["ingredients_tags"] = clean_series(df["ingredients_tags"].astype(str).where(df["ingredients_tags"].notna(), np.nan))
df["ingredients_text"] = df["ingredients_text"].astype("string").str.strip()
print(f"  ingredients_tags sauber gehalten ({df['ingredients_tags'].notna().sum():,})  ·  ingredients_text erhalten ({df['ingredients_text'].notna().sum():,})")

# Manufacturing Places — Tag-Spalte aus Rohtext auffuellen (wie Hauptnotebook)
manufacturing_before = df["manufacturing_places_tags"].notna().sum()
df["manufacturing_places_tags"] = df["manufacturing_places_tags"].fillna(df["manufacturing_places"])
merge_report("manufacturing_places_tags", manufacturing_before, df["manufacturing_places_tags"].notna().sum())
df = df.drop(columns=["manufacturing_places"])


# ════════════════════════════════════════════════════════════════════════════════
# TEIL B — Reproduktion der >6%-Spalten (Logik aus uber6prozent), Praefixe BEHALTEN
# ════════════════════════════════════════════════════════════════════════════════
section_header("Teil B  ·  ingredients_analysis_tags · nova_group · nutrient_levels_tags")

# Ingredients Analysis Tags — kommaseparierte en:-Tags; Junk weg, kleinschreiben, Kommas normalisieren
iat_before = df["ingredients_analysis_tags"].notna().sum()
df["ingredients_analysis_tags"] = (
    clean_series(df["ingredients_analysis_tags"]).str.lower().str.replace(r"\s*,\s*", ",", regex=True)
)
clean_report("ingredients_analysis_tags", iat_before, df["ingredients_analysis_tags"].notna().sum())

# NOVA Group — numerisch erzwingen, nur gueltige Gruppen 1-4 behalten (nullable Int)
nova_before = df["nova_group"].notna().sum()
_nova = pd.to_numeric(df["nova_group"], errors="coerce")
df["nova_group"] = _nova.where(_nova.isin([1, 2, 3, 4])).astype("Int64")
clean_report("nova_group", nova_before, df["nova_group"].notna().sum())

# Nutrient Levels Tags — wie ingredients_analysis_tags
nlt_before = df["nutrient_levels_tags"].notna().sum()
df["nutrient_levels_tags"] = (
    clean_series(df["nutrient_levels_tags"]).str.lower().str.replace(r"\s*,\s*", ",", regex=True)
)
clean_report("nutrient_levels_tags", nlt_before, df["nutrient_levels_tags"].notna().sum())


# ════════════════════════════════════════════════════════════════════════════════
# TEIL C — NEU: cities_tags · owner · traces · traces_tags · traces_en  (Praefixe BEHALTEN)
# ════════════════════════════════════════════════════════════════════════════════
section_header("Teil C (neu)  ·  cities_tags · owner · traces · traces_tags · traces_en")

# Tag-Spalten: Junk weg, kleinschreiben, Komma-Abstaende vereinheitlichen — en:/fr: bleiben erhalten
for col in ["cities_tags", "traces", "traces_tags", "traces_en"]:
    before = df[col].notna().sum()
    df[col] = clean_series(df[col]).str.lower().str.replace(r"\s*,\s*", ",", regex=True)
    clean_report(col, before, df[col].notna().sum())

# owner — ID-String: nur saeubern + Whitespace kollabieren; NICHT kleinschreiben, nicht kommasplitten
owner_before = df["owner"].notna().sum()
df["owner"] = clean_series(df["owner"]).str.replace(r"\s+", " ", regex=True).str.strip().replace("", pd.NA)
clean_report("owner", owner_before, df["owner"].notna().sum())


# ── Spalten umbenennen (nur die gemergten Survivors; cities_tags/traces*/owner behalten ihre Namen)
df = df.rename(columns={
    "packaging_en":              "packaging",
    "additives_en":              "additives",
    "ingredients_tags":          "ingredients",
    "manufacturing_places_tags": "manufacturing_places",
})

print(f"\n{'─' * _W}")
print(f"  Zwischenstand: {df.shape[1]} Spalten gesamt  |  {len(df):,} Zeilen")
print(f"{'─' * _W}")

In [ ]:
# Sample-Uebersicht der in diesem Notebook gereinigten Spalten
pd.set_option("display.max_colwidth", 80)
cols = ["product_name", "packaging", "additives", "ingredients", "ingredients_text",
        "manufacturing_places", "ingredients_analysis_tags", "nova_group", "nutrient_levels_tags",
        "cities_tags", "owner", "traces", "traces_tags", "traces_en"]
cols = [c for c in cols if c in df.columns]

uebersicht = pd.DataFrame({
    "dtype":    df[cols].dtypes.astype(str),
    "fuellung": (df[cols].notna().mean() * 100).round(1).astype(str) + " %",
    "beispiel": {c: (df[c].dropna().iloc[0] if df[c].notna().any() else None) for c in cols},
})
print(uebersicht, "\n")

# Pro Spalte die ersten 5 echten Werte (Praefixe sollten erhalten sein, z. B. en:milk)
n = 5
pd.DataFrame({c: df[c].dropna().head(n).reset_index(drop=True) for c in cols})